# v2 REAL + 에피소드 녹화 — 개인 의도 보정과 XYZ 배율

기존 strict JSON worker를 사용합니다. 처음에는 DRY RUN이고, PC에서 REAL RUN을 선택하면 실제 명령이 전송됩니다. 먼저 SIM에서 개인 보정의 수평/상하/대각선 동작을 확인하세요.

위치 매핑: `operator delta → C → XYZ gains → 20cm 작업범위 → IK`. Z를 고정하거나 전진 궤적으로 강제 투영하지 않습니다.
기준 자세는 로봇의 현재 EEF와 편한 컨트롤러 자세를 양손 grip 시점에 대응시킵니다. 좌표는 로봇 전방 +X, 왼쪽 +Y, 위 +Z입니다.


> **녹화 버전.** `real_vr_teleop.ipynb` 와 텔레옵 코드·설정이 같고, 녹화만 추가했습니다 (`package/episode_recorder.py`). 텔레옵 코드는 수정하지 않았습니다.
>
> - Orin: `worker_bringup_teleop` → `SG2_FIXED_QUEST=1 worker_outbound_meta` (관절 + 카메라 3대) → `SG2_FIXED_QUEST=1 SG2_ZMQ_SUB_IP=<PC IP> worker_inbound`
> - 실행하면 `SG2 Record` 창이 함께 뜹니다: `STOP` → `REC` 녹화 시작, `REC` → `STOP` 저장, `DEL` 녹화 중이면 버리고 아니면 마지막 에피소드 삭제
> - 창 제목에 상태와 수신 Hz (joint / action / 카메라별) 표시. action 은 REAL RUN 에서 양손 grip 으로 조작 중일 때만 올라갑니다
> - 저장: [LeRobot v3.0 데이터셋](https://huggingface.co/docs/lerobot/en/lerobot-dataset-v3) 형식으로 `SAVE_ROOT/TASK_NAME`에 직접 기록합니다 (`action`·`observation.state`는 `JOINT_NAMES` 순서, 카메라마다 `observation.images.<cam>` 비디오). 모든 카메라에서 첫 프레임이 와야 데이터셋 스키마(해상도)가 정해지므로, REC를 눌러도 카메라 프레임을 받기 전까지는 창이 자동으로 `STOP`으로 돌아갑니다 — 창 제목에 대기 상태가 표시됩니다.
>
> **Quest 컨트롤러 버튼** (XR 화면 안에서)
>
> | 버튼 | 동작 |
> |---|---|
> | 왼쪽 `X` | 녹화 시작 / 끝 (누를 때마다 전환) |
> | 왼쪽 `Y` 1초 | 삭제 — 녹화 중이면 버리고, 아니면 마지막 에피소드 삭제 |
> | 오른쪽 `A` (한 번 누름) | **Initial pose** 복귀 — 실행 창의 `Initial pose` 버튼 클릭과 완전히 동일합니다 (홀드 아님). 고정된 `worker_bringup` 자세(`BRINGUP_POSE`)까지 전체 경로를 검사한 뒤 10°/s로 복귀합니다 |
> | 오른쪽 `B` 1초 | **reset pose**: 양손 5cm 위로 + 바깥으로 5cm → `worker_shutdown` 처럼 접기 (4단계) → `worker_bringup` 초기 자세 (5단계). 도는 동안 조작 입력 무시 |
>
> A·B 모두 머리를 `head_joint1` 0.3 rad(약 17°) 아래로 내립니다 (`vr_buttons.HEAD_TILT`). 좌우 회전(`head_joint2`)은 실측값을 유지하며, 두 머리 관절의 HOME 허용오차는 정착 오차(~0.7°) 때문에 2°로 완화되어 있습니다.
>
> B 이동은 단계마다 경로 충돌 검사 후, 양손 위·바깥 이동과 첫 팔 정렬(`shutdown1`)까지 30°/s를 유지합니다. 도달 확인 후 `shutdown2`부터 뒤로 접기·앞으로 펴기는 팔 100°/s로 진행합니다. B 팔 속도는 일반 조작 속도와 별도로 적용하며, 머리·그리퍼 상한과 lift 속도는 유지합니다. 구버전 worker가 30/60°/s이면 해당 상한을 알리고 전체 접기·펴기를 완료합니다. 100°/s 적용에는 Orin의 `fixed_quest_protocol.py`와 `inbound.py`를 이번 버전으로 교체하고 inbound를 재시작해야 합니다(로그 `home 100.0 deg/s`). 상수만 바꾸면 이전 코드가 상한을 60°/s로 보고할 수 있습니다. 노트북 커널도 재시작하세요. 경로가 막히면 해당 경로의 명령은 보내지 않습니다. DRY RUN에서는 화면에서만 움직입니다. 진행 상태는 컨트롤 창 `버튼:` 줄에 표시됩니다.
> Meta(Oculus) 버튼과 왼쪽 메뉴 버튼은 시스템 전용이라 웹에서 읽을 수 없습니다.
>
> **베이스 이동 (키보드)**: `SG2 Base` 창에서 `Base command` 를 `ON` → **창을 클릭** → `W/S` 전후 · `A/D` 좌우 · `Q/E` 회전 · `SPACE` 즉시 정지.
> 속도 `SLOW` 0.09 m/s / `NORMAL` 0.18 / `FAST` 0.3 (회전 0.18 / 0.36 / 0.6 rad/s). 창이 포커스를 잃거나 OFF·종료하면 정지하고, 명령이 1초 끊기면 로봇이 스스로 멈춥니다.
> Orin `inbound.py` 가 베이스 전용 채널(:5562)을 받아야 합니다 (2026-09-19 이후 버전, 재시작 필요).

수동 관절 조정: 조작을 멈춘 뒤 관절과 절대 목표 각도(°)를 선택하고 **관절 미리보기 → 조정 실행**을 누릅니다. 같은 창의 **Cancel return**으로 중단하고, 조정 후 **Initial pose**로 복귀할 수 있습니다. 최초 HOME 목표는 보존됩니다.
그리퍼는 기본 60°/s, 새 bridge 사용 시 10~120°/s를 입력할 수 있습니다. REAL 적용 전 로봇 PC의 `inbound.py`와 `fixed_quest_protocol.py`를 동일 버전으로 교체하고 inbound/outbound를 재시작하세요. HOME과 수동 조정은 최대 10°/s입니다.


In [1]:
# Locate this standalone workspace from a notebook or repository directory.
from pathlib import Path
import runpy

_v2_candidates = [candidate for base in (Path.cwd(), *Path.cwd().parents)
                  for candidate in (base, base / "VR_teleoperation")]
_v2_root = next((candidate for candidate in _v2_candidates
                 if (candidate / ".project_v2").is_file()), None)
if _v2_root is None:
    raise RuntimeError("Open this notebook from the VR_teleoperation workspace.")
_v2_bootstrap = runpy.run_path(str(_v2_root / "package" / "init_project.py"))
for _v2_name in _v2_bootstrap["__all__"]:
    globals()[_v2_name] = _v2_bootstrap[_v2_name]


import numpy as np



[VR_teleoperation / ffw_sg2_vr_teleoperation] initialized.
 - PROJECT_DIR: /home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation


## 설정 — 사용자 SIM 확인을 반영한 v1 REAL 시험

사용자가 SIM 조작을 확인하고 REAL 시험을 요청한 v1 후보를 지정했습니다. 원본 accepted=false를 변경하지 않으며, 별도 SIM 확인 기록을 프로필·시연의 해시에 연결합니다. 다른 후보로 바뀌면 기존 확인 기록으로 로딩되지 않습니다.

XYZ 모두 0.5, 팔 속도 상한 60°/s, Initial pose 10°/s입니다. 기존 관절·충돌·속도·피드백 제한을 유지합니다. 기본 DRY RUN에서 연결과 피드백을 확인한 뒤 REAL RUN 버튼으로 실제 명령 전송을 시작합니다. 일반 통과 프로필을 사용할 때는 REAL_CANDIDATE_REVIEW_PATH=None으로 설정합니다.


In [2]:
from package.intent_teleop import IntentTeleopSettings, selected_profile, build_models, run_intent_teleop, tracking_mode_text
from package.episode_recorder import find_lan_ip
ROBOT_IP = "192.168.6.2"
settings = IntentTeleopSettings(
    backend="real",
    PROFILE_PATH=PROJECT_DIR / "calibration/diagnostics/v1.json",
    REAL_CANDIDATE_CAPTURE_PATH=PROJECT_DIR / "calibration/diagnostics/v1_trajectory.json",
    REAL_CANDIDATE_REVIEW_PATH=PROJECT_DIR / "calibration/reviews/v1-sim-reviewed-real-trial.json",
    CONFIRM_CALIBRATION_SETUP=True,  # 본인·Quest 방향·작업 위치를 확인한 후 True
    POSITION_SCALE=.5, POSITION_SCALE_Y=.5, POSITION_SCALE_Z=.5,
    MAX_JOINT_SPEED=np.deg2rad(100.),  # 실행 창에서 10~120 숫자 입력 + 적용
    TELEOP_RAMP_S=3.0,  # 시작/재그립 후 팔 속도 상한까지 ramp 시간 [초]
    MAX_GRIPPER_SPEED=np.deg2rad(120.),  # 그리퍼 전용 상한. 새 worker에서 10~120 숫자 입력 + 적용 (복귀 중 10°/s)
    IK_RECOVERY_TIMEOUT_S=2,  # IK 미수렴 시 최대 2초 유지·재시도 (0: 즉시 중단)
    NULL_REFERENCE="self_motion",  # 기존 IK 비교 모드에서만 사용
    POSTURE_IK_ENABLED=True,  # False: 기존 IK로 비교 (런타임 재시작)
    TELEOP_COLLISION_CHECK=False,  # Teleoperation OFF · Initial pose/Reset pose 충돌 검사 ON
    POSTURE_POSITION_TOLERANCE_M=.001,  # IK 후보 허용오차 1 mm
    POSTURE_ROTATION_TOLERANCE_DEG=.5,  # IK 후보 허용오차 0.5°
    POSTURE_ELBOW_LIMIT_DEG=85.,  # 초기 팔꿈치 방향에서 반대편으로 넘어가지 않음
    NULL_ANCHOR_ON_GRIP=False,  # 초기 팔 자세 기준은 첫 조작 시 실측으로 한 번만 저장
    ROTATION_DEADBAND_DEG=0.,  # 컨트롤러 상대 회전을 그대로 추종
    COLLISION_ESCAPE_MIN_M=.001,  # 자동 이동 시 기존 탈출 예외: 1~5mm 간격에서 시작보다 가까워지지 않는 경로 허용 (0.2mm 오차)
    QUEST_HOST=find_lan_ip(ROBOT_IP),  # 로봇과 같은 네트워크(192.168.6.x)의 PC IP 자동 탐색 (Wi-Fi/유선 모두)
    TRACKING_PAUSE_S=.1, ACCEPT_EMULATED_POSE=True,  # 시작 모드: 추정 위치 사용 (컨트롤러가 사라지면 0.1초 뒤 정지)
    TRACKING_MODE_BUTTONS=("emulated", "pause_0.5"),  # 실행 창 버튼: 추정 위치 사용 / 정지 0.5초
    ROBOT_IP=ROBOT_IP,
    DRY_RUN=True,
)
settings.validate()
print("IK 모드:", "초기 팔 자세 유지" if settings.POSTURE_IK_ENABLED else "기존 IK")
profile, matrices = selected_profile(settings)
print("개인 보정:", "미적용 — identity" if profile is None else profile["profile_id"])
print("PC IP:", settings.QUEST_HOST, "→ Quest: https://%s:%d/ | Orin: SG2_FIXED_QUEST=1 SG2_ZMQ_SUB_IP=%s worker_inbound" % (settings.QUEST_HOST, settings.QUEST_PORT, settings.QUEST_HOST))
print("XYZ:", settings.POSITION_SCALES["left"], "| 속도:", np.rad2deg(settings.MAX_JOINT_SPEED),
      "| 그리퍼:", np.rad2deg(settings.MAX_GRIPPER_SPEED), "| 기존 IK null 기준:", settings.NULL_REFERENCE)
print("사용자 SIM 확인 후 REAL 후보 시험 | 자동 검증 accepted:", profile["accepted"])
print("추적 손실 시작 모드:", tracking_mode_text({"tracking_pause_s": settings.TRACKING_PAUSE_S,
                                                 "accept_emulated": settings.ACCEPT_EMULATED_POSE}))

/home/vai/anaconda3/envs/ri_motion_v5_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


IK 모드: 초기 팔 자세 유지
개인 보정: 51738a755a527d0c
PC IP: 192.168.6.103 → Quest: https://192.168.6.103:8443/ | Orin: SG2_FIXED_QUEST=1 SG2_ZMQ_SUB_IP=192.168.6.103 worker_inbound
XYZ: [0.5, 0.5, 0.5] | 속도: 100.0 | 그리퍼: 119.99999999999999 | 기존 IK null 기준: self_motion
사용자 SIM 확인 후 REAL 후보 시험 | 자동 검증 accepted: False
추적 손실 시작 모드: 추정 위치 사용 (완전 소실 시 정지 0.1초)


## 모델 준비 확인 (통신 없음)

기존 FK·IK와 관절/충돌 검사를 그대로 사용합니다. 아래 셀은 모델만 확인합니다.


In [3]:
models = build_models(settings)
from utils import get_sg2_wrist_control_T
for hand in ("left", "right"):
    print(hand, "EEF (m):", get_sg2_wrist_control_T(models["env"], hand)[:3, 3])
del models


MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_real_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_real_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_real_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_real_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_real_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_real_scene.xml].
Command joints: 19 | Collision pairs: 499
No robot connection or commands in this setup cell.
left EEF (m): [0.468004 0.2275   1.100596]
right EEF (m): [ 0.468004 -0.2275    1.100596]


## 실행

[Worker 실행 안내](../docs/WORKER_SETUP.md)에 따라 bringup/outbound/inbound를 실행하세요.

- 다른 Quest 수신 노트북을 STOP하여 8443 포트를 비우세요.
- 양손 grip으로 시작하고, 한쪽이라도 놓으면 양팔이 멈춥니다.
- XYZ는 grip을 모두 놓고 숫자 입력 후 적용합니다. 새 grip 기준에서 반영되므로 목표가 갑자기 바뀌지 않습니다.
- 속도는 실행 중 입력 후 적용할 수 있습니다. 감속 시 이전 명령 ACK를 기다릴 수 있습니다.
- 그리퍼 속도는 팔 속도와 별개로 실행 중 10~60°/s로 적용할 수 있습니다. 로봇 bridge가 그리퍼를 60°/s로 제한하므로 그 이상은 입력할 수 없고, Initial pose 복귀 중에는 복귀 속도를 따릅니다.
- Initial pose는 Quest/그립 없이 고정된 `worker_bringup` 자세(`BRINGUP_POSE`)로 10°/s 복귀합니다 — 이번 실행에서 로봇이 어떤 자세로 있었는지와 무관합니다. 머리는 `head_joint1`이 0.3 rad(약 17°) 아래를 향하고, 좌우 회전(`head_joint2`)만 실측값을 유지합니다. 오른쪽 컨트롤러 `A` 버튼을 한 번 누르면 같은 동작을 요청합니다 (버튼 클릭과 동일 — 홀드 아님).
- 누르는 순간 복귀 전체 경로를 기존 관절·5mm 충돌 검사로 확인합니다. 직선 경로가 막히면 J3·J5 먼저 풀기 → 전완·손목 먼저 풀기 → 오른팔 먼저 → 왼팔 먼저 순서로 단계 경로를 찾고, 명령은 확인한 경로 위로만 보냅니다. 모든 경로가 막히면 아무 명령도 보내지 않으므로, VR 조작으로 팔 사이를 벌린 뒤 다시 누르세요. 중간에 멈춘 복귀도 다시 누르면 현재 자세에서 새로 검사합니다.
- **추적 손실 모드**: 기본은 `추정 위치 사용`입니다. 한 손의 광학 추적이 끊겨도 Quest 추정 위치로 조작을 이어가고, 컨트롤러가 아예 사라지면 0.1초 뒤 정지합니다. 추정 위치는 손실 순간 수 cm~십수 cm 튈 수 있습니다.
- 실행 창에서 `정지 0.5초`로 바꾸면 추적 손실 중 현재 명령을 최대 0.5초 유지하고, 그 안에 추적이 돌아오면 같은 grip 기준에서 이어갑니다. 모드 전환은 양손 grip을 놓은 상태에서 적용됩니다.
- 두 모드 모두 한 프레임 20cm 초과 점프·grip 해제·피드백/실행 오류는 정지하고, grip당 20cm 작업범위와 속도·충돌 제한은 그대로입니다.
- 일시적인 IK 미수렴은 양팔·그리퍼의 현재 명령을 최대 1초 유지하며 최신 목표로 재시도합니다. 복구되면 같은 grip 기준에서 조작을 이어갑니다. 화면에 경과 시간과 재시도 횟수가 표시되며, 시간 초과·그립 해제·피드백 이상은 정지합니다.
- 팔꿈치 양옆의 두 회전(Joint 3·5)이 조작과 무관하게 감기지 않도록, EEF가 볼 수 없는 자세 성분만 시작 자세 쪽으로 유지합니다. 목표 추종에 쓰이는 성분은 기존과 같이 IK가 결정합니다.
- 로그는 `.cache/fixed_quest/recordings/*-intent-real.jsonl`에 기록됩니다.

In [4]:
# 녹화 설정 (통신 없음 — 수신 스레드는 실행 셀에서 시작)
from PyQt5.QtWidgets import QApplication
from ri_motion_v5_package.qt import use_pyqt_platform_plugins
from package.episode_recorder import VREpisodeRecorder, RecordControl, list_episodes, DEFAULT_CAMERAS
use_pyqt_platform_plugins()  # Linux: cv2 (imported by episode_recorder) points Qt at its own plugin dir
app = QApplication.instance() or QApplication([])

SAVE_ROOT = Path.home() / "aiworker_data"
TASK_NAME = "pick_and_place"
recorder = VREpisodeRecorder(
    robot_ip         = settings.ROBOT_IP,
    repo_id          = "local/%s" % TASK_NAME,   # Hub에 올리지 않는 로컬 데이터셋 이름
    root             = SAVE_ROOT / TASK_NAME,    # LeRobot v3.0 데이터셋 루트 (data/ videos/ meta/)
    hz               = 30,
    max_sec          = 120,          # 에피소드당 안전 타임아웃
    cameras          = DEFAULT_CAMERAS,   # {"head": 5570, "wrist_left": 5571, "wrist_right": 5572}
    command_endpoint = "tcp://127.0.0.1:%d" % settings.ROBOT_COMMAND_PORT,
    task             = TASK_NAME,    # LeRobot 프레임마다 붙는 task 설명
)
print("save to:", SAVE_ROOT / TASK_NAME, "| 이미", len(list_episodes(SAVE_ROOT / TASK_NAME)), "개")

save to: /home/vai/aiworker_data/pick_and_place | 이미 33 개


In [5]:
from package.vr_buttons import ControllerButtonActions
recorder.start_thread()
rec_control = RecordControl(recorder)
buttons = ControllerButtonActions(record_control=rec_control)   # X 녹화 · Y(1초) 삭제 · A Initial pose · B(1초) reset pose
try:
    log_path = run_intent_teleop(settings, on_tick=buttons)
finally:
    recorder.close()      # 녹화 중이었다면 저장, LeRobot 데이터셋 finalize
    rec_control.close()

[rec] ready (3 cameras, 33 saved)
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_real_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_real_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_real_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_real_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_real_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_real_scene.xml].
Command joints: 19 | Collision pairs: 499
No robot connection or commands in this setup cell.
Quest Browser: https://192.168.6.103:8443/
Robot workers: SG2_FIXED_QUEST=1 | inbound SG2_ZMQ_SUB_IP=192.168.6.103
SHADOW | 양손 grip으로 

## 수집 결과 확인

In [6]:
# 저장된 에피소드는 LeRobot v3.0 데이터셋의 meta/episodes/*.parquet 에서 읽습니다.
# (recorder.close() 가 finalize 하며 flush 하므로, 실행 셀을 STOP 한 뒤 확인하세요)
eps = list_episodes(SAVE_ROOT / TASK_NAME)
print(f"{len(eps)} episodes in {SAVE_ROOT / TASK_NAME}")
tot = 0.0
for e in eps:
    duration = e["length"] / recorder.hz
    tot += duration
    print(f"  episode_{e['episode_index']:06d}: {duration:5.1f}s  {e['length']:5d} frames  task={e['tasks']}")
print(f"total {tot/60:.1f} min")

33 episodes in /home/vai/aiworker_data/pick_and_place
  episode_000000:  26.3s    789 frames  task=['pick_and_place']
  episode_000001:  20.3s    610 frames  task=['pick_and_place']
  episode_000002:  27.0s    810 frames  task=['pick_and_place']
  episode_000003:  24.3s    729 frames  task=['pick_and_place']
  episode_000004:  36.3s   1089 frames  task=['pick_and_place']
  episode_000005:  25.9s    777 frames  task=['pick_and_place']
  episode_000006:  24.3s    729 frames  task=['pick_and_place']
  episode_000007:  23.0s    691 frames  task=['pick_and_place']
  episode_000008:  19.3s    579 frames  task=['pick_and_place']
  episode_000009:  22.9s    687 frames  task=['pick_and_place']
  episode_000010:  22.6s    678 frames  task=['pick_and_place']
  episode_000011:  20.2s    607 frames  task=['pick_and_place']
  episode_000012:  22.8s    684 frames  task=['pick_and_place']
  episode_000013:  19.7s    591 frames  task=['pick_and_place']
  episode_000014:  21.7s    651 frames  task=['pic